# Lab 23.4: Same Model, Two Loaders

Chapter 23, Choosing, Measuring and Using XAI Tools. Run the cells in order; nothing needs to be edited.

A language-model interpretability library sits between you and the model. Before you trust its activations, check them against a second implementation of the same model. Here GPT-2 small is loaded three ways: with Hugging Face `transformers` (the reference implementation), with TransformerLens's default loader (which folds LayerNorm weights and centers some weights to make analysis easier), and with TransformerLens's no-processing loader. We compare what each returns.

In [1]:
import time, torch, numpy as np, pandas as pd
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
import transformers, transformer_lens
from transformer_lens import HookedTransformer
torch.manual_seed(0)
print("torch", torch.__version__, "| transformers", transformers.__version__)
try:
    from importlib.metadata import version; print("transformer-lens", version("transformer-lens"))
except Exception as e:
    print(e)

torch 2.6.0 | transformers 4.57.6
transformer-lens 2.18.0


## 1. Load the model three ways

In [2]:
tok = GPT2TokenizerFast.from_pretrained("gpt2")
t = time.time(); hf = GPT2LMHeadModel.from_pretrained("gpt2").eval(); t_hf = time.time() - t
t = time.time(); tl = HookedTransformer.from_pretrained("gpt2", device="cpu"); t_tl = time.time() - t
t = time.time(); raw = HookedTransformer.from_pretrained_no_processing("gpt2", device="cpu"); t_raw = time.time() - t
print(f"load seconds: transformers {t_hf:.1f}, TransformerLens default {t_tl:.1f}, no processing {t_raw:.1f}")

Loaded pretrained model gpt2 into HookedTransformer


Loaded pretrained model gpt2 into HookedTransformer
load seconds: transformers 0.3, TransformerLens default 1.2, no processing 1.1


## 2. Compare outputs and activations on four prompts

For each prompt we record the largest absolute difference from the Hugging Face reference in (a) next-token log-probabilities, (b) raw logits, (c) the logit difference between two candidate answers, and (d) the residual stream entering layers 0, 6 and 11.

In [3]:
prompts = [
    ("The bank declined the loan because the applicant", " had", " was"),
    ("When Mary and John went to the store, John gave a drink to", " Mary", " John"),
    ("The flight was delayed because of the", " weather", " crew"),
    ("The patient was admitted to the hospital with a", " fever", " broken"),
]
rows = []
for text, a, b in prompts:
    ids = tok(text, return_tensors="pt").input_ids
    ia, ib = tok(a).input_ids[0], tok(b).input_ids[0]
    with torch.no_grad():
        o = hf(ids, output_hidden_states=True)
        lg_tl, c_tl = tl.run_with_cache(ids)
        lg_raw, c_raw = raw.run_with_cache(ids)
    ref_logit = o.logits[0, -1]
    for name, lg, c in [("TL default", lg_tl, c_tl), ("TL no processing", lg_raw, c_raw)]:
        lgl = lg[0, -1]
        r = {"prompt": text[:28] + "...", "loader": name,
             "top token": tok.decode(lgl.argmax()),
             "logprob diff": (torch.log_softmax(ref_logit, -1) - torch.log_softmax(lgl, -1)).abs().max().item(),
             "logit diff": (ref_logit - lgl).abs().max().item(),
             "logit-difference diff": abs((ref_logit[ia] - ref_logit[ib]) - (lgl[ia] - lgl[ib])).item()}
        for L in (0, 6, 11):
            r[f"resid L{L}"] = (o.hidden_states[L][0] - c[f"blocks.{L}.hook_resid_pre"][0]).abs().max().item()
        rows.append(r)
    print(f"{text!r}: reference top token {tok.decode(ref_logit.argmax())!r}")
df = pd.DataFrame(rows)
pd.set_option("display.width", 160); pd.set_option("display.float_format", lambda v: f"{v:.2g}")
df

'The bank declined the loan because the applicant': reference top token ' had'
'When Mary and John went to the store, John gave a drink to': reference top token ' Mary'


'The flight was delayed because of the': reference top token ' weather'
'The patient was admitted to the hospital with a': reference top token ' minor'


,prompt,loader,top token,logprob diff,logit diff,logit-difference diff,resid L0,resid L6,resid L11
0,The bank declined the loan b...,TL default,had,0.00029,1.2e+02,0.00012,0.0068,4.6,4.9
1,The bank declined the loan b...,TL no processing,had,8.2e-05,0.00012,1.5e-05,0,0.00049,0.00049
2,When Mary and John went to t...,TL default,Mary,0.00019,99,1.9e-06,0.0068,4.6,4.8
3,When Mary and John went to t...,TL no processing,Mary,7.4e-05,9.9e-05,3.1e-05,0,0.00024,0.00024
4,The flight was delayed becau...,TL default,weather,0.00018,1e+02,4.4e-05,0.0068,4.6,4.9
5,The flight was delayed becau...,TL no processing,weather,6.7e-05,6.9e-05,1.5e-05,0,0.00049,0.00049
6,The patient was admitted to ...,TL default,minor,0.00034,1.3e+02,9.3e-05,0.0068,4.6,4.9
7,The patient was admitted to ...,TL no processing,minor,7.8e-05,9.2e-05,5.3e-05,0,0.00049,0.00049


**Checkpoint 1.** All three loaders give the same top token for every prompt. Log-probabilities agree to within about 0.0004 for both TransformerLens loaders.

**Checkpoint 2.** Raw logits from the default loader differ from the reference by tens of units (about 99 to 130 across the four prompts in our run), yet log-probabilities and the logit difference between two answers agree. The default loader centers the unembedding, which shifts every logit at a position by the same constant: softmax and differences are unchanged, raw values are not.

**Checkpoint 3.** The residual stream from the default loader differs from the reference by about 4.6 to 4.9 units at layers 6 and 11 (it is centered); the no-processing loader matches to within about 0.001.

## 3. Which quantities are safe to compare across tools?

In [4]:
summary = df.groupby("loader")[["logprob diff", "logit diff", "logit-difference diff", "resid L0", "resid L6", "resid L11"]].max()
summary.T

loader,TL default,TL no processing
logprob diff,0.00034,8.2e-05
logit diff,1.3e+02,0.00012
logit-difference diff,0.00012,5.3e-05
resid L0,0.0068,0
resid L6,4.6,0.00049
resid L11,4.9,0.00049


## Questions

1. A colleague reports "the logit for ' Mary' is 14.2" from one tool and you see 112 in another. Are both right? Which quantity should the two of you compare instead?
2. You plan to train a linear probe (Chapter 15) on residual-stream activations and then apply it to activations from a different library. What can go wrong, and how would this notebook have caught it?
3. TransformerLens 4.0 removed `HookedTransformer.from_pretrained`; its new default loader keeps the raw Hugging Face weights unless a compatibility mode is switched on. Write the two-line note you would add to a tool memo so that results from 2.x and 4.x are never mixed silently.
4. Which of the measurements in this notebook would you re-run every time a library version changes?